# النوتبوك ٧: Unified Evaluation Framework + Baseline Recommenders (النسخة ٢)

**الهدف:** نبني إطار تقييم موحّد بتستعمله كل الأنظمة (هون وبنوتبوك ٨ و٩)، ونجرّب عليه أول نظامين مع خط مقارنة عشوائي.

**المدخلات:**
- `recipes_labeled.pkl` (نوتبوك ٥)
- `interactions_train.pkl`، `interactions_test.pkl`، `recipe_stats_train.csv` (نوتبوك ٦)

**ما الجديد بالنسخة ٢ (9 نوفمبر):**
- التقييم على **كل** مستخدمات الاختبار (2,392 بعد فلتر الكتالوج)، لأنو العينة 1,000 كانت صغيرة (Popularity لاقى 14 بس).
- **طريقتين لملف الذوق بـ Content-Based** منقارنهن: المتوسط (centroid) وأعلى تشابه (max-similarity).

**القرارات المتفق عليها (5 نوفمبر):**
1. **التدريب على كل التفاعلات**، بس **الاقتراح والتقييم على الكتالوج** (good + fair بدون implausible). وصفات الاختبار اللي برا الكتالوج بتنشال، والمستخدمة اللي ما بيضل إلها شي بالاختبار بتطلع.
2. **المقاييس:** HitRate@10، NDCG@10، Coverage. الوصفات اللي المستخدمة شافتها بالتدريب ما بتنقترح.
3. **كل مستخدمات الاختبار** (`EVAL_SAMPLE = None`). إذا لزم للسرعة، فيكي تحطي رقم (العينة بـ SEED=42).
4. **خط مقارنة Random** لنتأكد إنو الأنظمة أحسن من الحظ.
5. **Popularity:** الـ `weighted_rating` من التدريب (m=5).
6. **Content-Based:** TF-IDF على المكونات (`parts_norm`)، cosine، بطريقتين لملف الذوق (A: المتوسط، B: أعلى تشابه). اللي ما عندها ولا وصفة محبوبة بالتدريب بترجع للـ Popularity.

**المخرجات:**
- `catalog_ids.csv`: وصفات الكتالوج
- `eval_test_catalog.pkl`: الاختبار بعد فلتر الكتالوج
- `eval_users_sample.csv`: المستخدمات اللي تقيّمنا عليهن
- `tfidf_ingredients.npz` + `tfidf_recipe_ids.csv`: مصفوفة TF-IDF (للـ Hybrid والتطبيق)
- `results_07.csv`: جدول النتائج

## 0. الإعدادات

In [16]:
import os
import html

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

# ---- المدخلات (نصوص) ----
RECIPES_PKL = "recipes_labeled.pkl"
TRAIN_PKL = "interactions_train.pkl"
TEST_PKL = "interactions_test.pkl"
STATS_CSV = "recipe_stats_train.csv"

# ---- المخرجات (نصوص) ----
CATALOG_CSV = "catalog_ids.csv"
EVAL_TEST_PKL = "eval_test_catalog.pkl"
EVAL_USERS_CSV = "eval_users_sample.csv"
TFIDF_NPZ = "tfidf_ingredients.npz"
TFIDF_IDS_CSV = "tfidf_recipe_ids.csv"
RESULTS_CSV = "results_07.csv"

# ---- القرارات ----
K = 10                              # أعلى 10
CATALOG_QUALITY = ["good", "fair"]  # الكتالوج
IMPLAUSIBLE_COL = None              # None = بيتلاقى تلقائياً؛ أو اكتبي اسم العمود
EVAL_SAMPLE = None                  # None = كل مستخدمات الاختبار؛ أو رقم لعينة
SEED = 42
BATCH = 500                         # عدد المستخدمات بكل دفعة حساب (للذاكرة)

for p in [RECIPES_PKL, TRAIN_PKL, TEST_PKL, STATS_CSV]:
    assert isinstance(p, str), f"{p!r} لازم يكون نص (path string)"
    assert os.path.exists(p), f"ما لقيت {p}: عدّلي المسار"

## 1. تحميل البيانات

In [17]:
recipes = pd.read_pickle(RECIPES_PKL).reset_index(drop=True)
recipes["RecipeId"] = recipes["RecipeId"].astype("int64")
recipes["Name"] = recipes["Name"].astype(str).map(html.unescape)

train = pd.read_pickle(TRAIN_PKL)
test = pd.read_pickle(TEST_PKL)
stats = pd.read_csv(STATS_CSV)
for df_ in (train, test):
    df_["user_id"] = df_["user_id"].astype("int64")
    df_["recipe_id"] = df_["recipe_id"].astype("int64")
stats["recipe_id"] = stats["recipe_id"].astype("int64")

assert recipes["RecipeId"].is_unique, "في RecipeId مكرر"
for c in ["parts_norm", "quality", "Name"]:
    assert c in recipes.columns, f"العمود {c} مش موجود بـ {RECIPES_PKL}"

print(f"recipes: {len(recipes):,} | train: {len(train):,} | test: {len(test):,} | stats: {len(stats):,}")

recipes: 42,736 | train: 146,775 | test: 17,460 | stats: 23,328


## 2. الكتالوج

الوصفات اللي التطبيق بيقترح منها: جودة good أو fair، وبدون implausible.

In [18]:
if IMPLAUSIBLE_COL is None:
    imp_cands = [c for c in recipes.columns if "implaus" in c.lower()]
    print("أعمدة implausible المحتملة:", imp_cands)
    assert imp_cands, "ما لقيت عمود implausible: اكتبي اسمه بـ IMPLAUSIBLE_COL بالإعدادات"
    IMPLAUSIBLE_COL = imp_cands[0]
print("using:", IMPLAUSIBLE_COL)

is_implausible = recipes[IMPLAUSIBLE_COL].fillna(False).astype(bool)
in_catalog = recipes["quality"].isin(CATALOG_QUALITY) & ~is_implausible

print(pd.crosstab(recipes["quality"], is_implausible.rename("implausible"), margins=True))

catalog_ids = np.sort(recipes.loc[in_catalog, "RecipeId"].to_numpy())
n_cat = len(catalog_ids)
cat_set = set(catalog_ids)
cat_index = pd.Series(np.arange(n_cat), index=catalog_ids)   # recipe_id -> رقم العمود بالكتالوج
print(f"\nالكتالوج: {n_cat:,} وصفة")

أعمدة implausible المحتملة: ['implausible']
using: implausible
implausible  False   True    All
quality                         
fair          3938    739   4677
good          8437   1345   9782
poor          8071   2519  10590
unverified   12145   5542  17687
All          32591  10145  42736

الكتالوج: 12,375 وصفة


## 3. الاختبار بعد فلتر الكتالوج

- منشيل من الاختبار الوصفات اللي برا الكتالوج.
- المستخدمة اللي ما بيضل إلها ولا وصفة بالاختبار بتطلع من التقييم.
- **cold items:** وصفات اختبار ما إلها ولا تفاعل بالتدريب. Popularity والـ CF ما بيقدروا يلاقوهن، Content-Based بيقدر.

In [19]:
test_cat = test[test["recipe_id"].isin(cat_set)].reset_index(drop=True)
eval_users_all = np.sort(test_cat["user_id"].unique())

cold = ~test_cat["recipe_id"].isin(set(train["recipe_id"]))
filter_summary = pd.DataFrame({
    "before (all recipes)": [len(test), test["user_id"].nunique(), test["recipe_id"].nunique()],
    "after (catalog only)": [len(test_cat), len(eval_users_all), test_cat["recipe_id"].nunique()],
}, index=["test interactions", "eval users", "test recipes"])
filter_summary["kept_%"] = (filter_summary.iloc[:, 1] / filter_summary.iloc[:, 0] * 100).round(1)
print(filter_summary)
print(f"\ncold test items بالكتالوج: {cold.sum():,} من {len(test_cat):,} ({cold.mean()*100:.1f}%)")
print(f"وصفات اختبار لكل مستخدمة: median {test_cat.groupby('user_id').size().median():.0f}")

                   before (all recipes)  after (catalog only)  kept_%
test interactions                 17460                  4873    27.9
eval users                         4852                  2392    49.3
test recipes                       8337                  2299    27.6

cold test items بالكتالوج: 344 من 4,873 (7.1%)
وصفات اختبار لكل مستخدمة: median 1


## 4. العينة

العينة بتنسحب مرة وحدة وبتنحفظ، فكل الأنظمة (هون وبالنوتبوكات الجاية) بتتقيّم على **نفس المستخدمات**.

In [20]:
rng = np.random.default_rng(SEED)
if EVAL_SAMPLE is not None and EVAL_SAMPLE < len(eval_users_all):
    eval_users = np.sort(rng.choice(eval_users_all, size=EVAL_SAMPLE, replace=False))
    sample_label = f"sample {EVAL_SAMPLE}"
else:
    eval_users = eval_users_all
    sample_label = "all"
n_eval = len(eval_users)
u_pos = pd.Series(np.arange(n_eval), index=eval_users)       # user_id -> رقم السطر
print(f"eval users: {n_eval:,} ({sample_label})")

# الوصفات المخبّاية لكل مستخدمة (أرقام أعمدة الكتالوج)
tc = test_cat[test_cat["user_id"].isin(u_pos.index)]
relevant = (tc.assign(ci=cat_index.loc[tc["recipe_id"]].to_numpy())
              .groupby("user_id")["ci"].apply(np.array).to_dict())

# الوصفات اللي شافتها بالتدريب (كل التفاعلات، مش بس المحبوبة): ما بتنقترح
tr = train[train["user_id"].isin(u_pos.index) & train["recipe_id"].isin(cat_set)]
seen = (tr.assign(ci=cat_index.loc[tr["recipe_id"]].to_numpy())
          .groupby("user_id")["ci"].apply(np.array).to_dict())

print(f"وصفات مشاهدة بالتدريب لكل مستخدمة (ضمن الكتالوج): median {np.median([len(seen.get(u, [])) for u in eval_users]):.0f}")

eval users: 2,392 (all)
وصفات مشاهدة بالتدريب لكل مستخدمة (ضمن الكتالوج): median 3


## 5. إطار التقييم الموحّد

كل نظام بيعطي **درجة لكل وصفة بالكتالوج** لكل مستخدمة (`score_fn`). الإطار بيعمل الباقي بنفس الطريقة للكل:
1. بيشيل الوصفات المشاهدة بالتدريب.
2. بياخد أعلى 10.
3. بيحسب:
   - **HitRate@10:** نسبة المستخدمات اللي طلعتلهن وصفة مخبّاية وحدة على الأقل بالعشرة.
   - **NDCG@10:** متل HitRate بس بيعطي نقاط أكتر للوصفة الصح كل ما كانت أعلى بالترتيب (من 0 لـ 1).
   - **Coverage:** نسبة وصفات الكتالوج اللي طلعت ولو مرة بالاقتراحات (التنوّع).
- `hit_users`: عدد المستخدمات اللي لاقالهن النظام وصفة صح (لنعرف إذا الفرق بين الأنظمة كبير ولا صدفة).

`score_fn(rows)` بتاخد أرقام أسطر المستخدمات (من `eval_users`) وبترجع مصفوفة (عدد المستخدمات × عدد وصفات الكتالوج).

In [21]:
_disc = 1.0 / np.log2(np.arange(2, K + 2))
_train_ids = set(train["recipe_id"])
cold_ci = cat_index.loc[[r for r in catalog_ids if r not in _train_ids]].to_numpy()   # وصفات كتالوج بلا تفاعل بالتدريب


def recommend(score_fn):
    """أعلى K وصفة لكل مستخدمة بالعينة (أرقام أعمدة الكتالوج)، بعد شيل المشاهدة."""
    out = np.empty((n_eval, K), dtype=np.int64)
    for start in range(0, n_eval, BATCH):
        rows = np.arange(start, min(start + BATCH, n_eval))
        S = np.array(score_fn(rows), dtype=np.float32, copy=True)
        for r_, row in enumerate(rows):
            s_ = seen.get(eval_users[row])
            if s_ is not None:
                S[r_, s_] = -np.inf
        top = np.argpartition(-S, K, axis=1)[:, :K]
        order_ = np.argsort(-np.take_along_axis(S, top, axis=1), axis=1)
        out[rows] = np.take_along_axis(top, order_, axis=1)
    return out


def evaluate(recs, name):
    hits = np.zeros(n_eval, dtype=bool)
    ndcg = np.zeros(n_eval)
    cold_hit_count = 0
    for row, u in enumerate(eval_users):
        rel = relevant[u]
        h = np.isin(recs[row], rel)
        hits[row] = h.any()
        cold_hit_count += int(np.isin(recs[row][h], cold_ci).sum())
        ndcg[row] = (h * _disc).sum() / _disc[:min(len(rel), K)].sum()
    return {"system": name,
            f"HitRate@{K}": round(float(hits.mean()), 4),
            f"NDCG@{K}": round(float(ndcg.mean()), 4),
            "Coverage_%": round(len(np.unique(recs)) / n_cat * 100, 2),
            "hit_users": int(hits.sum()),
            "cold_hits": cold_hit_count,
            "eval_users": sample_label}


def show_recs(recs, row, n=K):
    ids = catalog_ids[recs[row][:n]]
    names = recipes.set_index("RecipeId").loc[ids, "Name"]
    hit = np.isin(recs[row][:n], relevant[eval_users[row]])
    return pd.DataFrame({"recipe_id": ids, "Name": names.to_numpy(), "hidden_hit": hit})


results = []

## 6. خط المقارنة: Random

درجات عشوائية. أي نظام ما بيغلبه ما إلو قيمة.

In [22]:
rand_rng = np.random.default_rng(SEED)
recs_random = recommend(lambda rows: rand_rng.random((len(rows), n_cat)))
results.append(evaluate(recs_random, "Random"))
results[-1]

{'system': 'Random',
 'HitRate@10': 0.0021,
 'NDCG@10': 0.0005,
 'Coverage_%': 85.6,
 'hit_users': 5,
 'cold_hits': 0,
 'eval_users': 'all'}

## 7. Popularity

نفس الترتيب لكل المستخدمات: الـ `weighted_rating` من التدريب (Bayesian، m=5). إذا تساوى، بيتقدّم الأكتر تفاعلات. الوصفات اللي ما إلها ولا تفاعل بالتدريب بتنحط بالآخر.

In [23]:
st = stats.set_index("recipe_id").reindex(catalog_ids)
wr = st["weighted_rating"].fillna(-1).to_numpy()
ni = st["n_interactions"].fillna(0).to_numpy()
pop_order = np.lexsort((ni, wr))                 # تصاعدي: الأضعف أولاً
pop_score = np.empty(n_cat, dtype=np.float32)
pop_score[pop_order] = np.arange(n_cat)          # الأعلى = الأشهر

print(f"وصفات الكتالوج اللي إلها تفاعل بالتدريب: {np.isfinite(st['n_interactions']).sum():,} من {n_cat:,}")

top_pop = catalog_ids[pop_order[::-1][:15]]
(st.loc[top_pop, ["weighted_rating", "n_ratings", "n_interactions"]]
   .join(recipes.set_index("RecipeId")[["Name", "RecipeCategory"]]))

وصفات الكتالوج اللي إلها تفاعل بالتدريب: 6,641 من 12,375


,weighted_rating,n_ratings,n_interactions,Name,RecipeCategory
recipe_id,,,,,
52285,4.962932,44.0,50.0,Cream Cheese Flan,Dessert
141488,4.952202,33.0,36.0,Snickerdoodles,Dessert
244193,4.933106,67.0,71.0,Absolutely the Best New York Cheesecake (Gluten-Free),Cheesecake
68670,4.924319,19.0,19.0,Italian Style Skillet Zucchini,Vegetable
102451,4.921029,18.0,18.0,Whipped Cauliflower Dijon,Cauliflower
113193,4.919533,30.0,31.0,"Sweet, Buttery Rolls (Bread Machine Recipe)",Yeast Breads
25779,4.917439,17.0,20.0,Cream Cheese Icing (Mrs. Echard Icing),Dessert
404470,4.917439,17.0,18.0,Fluffy Italian Bread,Yeast Breads
198785,4.917439,17.0,18.0,Pumpkin Cinnamon Rolls,Yeast Breads


In [24]:
recs_pop = recommend(lambda rows: np.broadcast_to(pop_score, (len(rows), n_cat)))
results.append(evaluate(recs_pop, "Popularity"))
results[-1]

{'system': 'Popularity',
 'HitRate@10': 0.0134,
 'NDCG@10': 0.0043,
 'Coverage_%': 0.11,
 'hit_users': 32,
 'cold_hits': 0,
 'eval_users': 'all'}

## 8. Content-Based (TF-IDF على المكونات)

1. كل وصفة بتصير vector: كل مكون (متل `chicken breast`) كلمة وحدة. TF-IDF بيعطي وزن أكبر للمكونات النادرة.
2. الـ TF-IDF محسوب على كل الـ 42,736 وصفة (مش بس الكتالوج)، لأن المستخدمة ممكن تكون حبّت وصفة برا الكتالوج وبرضو بتعبّر عن ذوقها.
3. منستعمل الوصفات المحبوبة (4+) بالتدريب، بطريقتين:
   - **A. المتوسط (centroid):** ملف ذوق واحد = متوسط vectors الوصفات المحبوبة، والدرجة = cosine بينه وبين كل وصفة.
   - **B. أعلى تشابه (max-similarity):** منقارن كل وصفة بالكتالوج مع **كل** وصفة محبوبة لحال، والدرجة = أعلى تشابه. هيك الأذواق المختلفة (شوربة وكيكة) ما بتذوب ببعض.

In [25]:
vec = TfidfVectorizer(analyzer=lambda parts: list(parts), lowercase=False)
X = vec.fit_transform(recipes["parts_norm"])          # كل سطر مطبّع (L2)، فالضرب = cosine
rid_to_row = pd.Series(np.arange(len(recipes)), index=recipes["RecipeId"])
X_cat = X[rid_to_row.loc[catalog_ids].to_numpy()]
print(f"TF-IDF: {X.shape[0]:,} وصفة × {X.shape[1]:,} مكون")

idf = pd.Series(vec.idf_, index=vec.get_feature_names_out()).sort_values()
print("\nأشيع المكونات (أقل وزن):", list(idf.index[:8]))
print("أندر المكونات (أعلى وزن):", list(idf.index[-8:]))

TF-IDF: 42,736 وصفة × 481 مكون

أشيع المكونات (أقل وزن): ['salt', 'butter', 'sugar', 'all purpose flour', 'egg', 'vanilla extract', 'baking powder', 'water']
أندر المكونات (أعلى وزن): ['light soy sauce', 'hoisin sauce', 'lemongrass', 'ghee', 'sake', 'bacon', 'sausage', 'ham']


In [26]:
lt = train[train["liked"] & train["user_id"].isin(u_pos.index) & train["recipe_id"].isin(rid_to_row.index)]
A = sparse.csr_matrix(
    (np.ones(len(lt)), (u_pos.loc[lt["user_id"]].to_numpy(), rid_to_row.loc[lt["recipe_id"]].to_numpy())),
    shape=(n_eval, len(recipes)))
n_liked_tr = np.asarray(A.sum(axis=1)).ravel()
P = sparse.diags(1.0 / np.maximum(n_liked_tr, 1)) @ A @ X     # ملف الذوق = المتوسط
P = P.tocsr()
no_profile = n_liked_tr == 0
print(f"مستخدمات بدون ولا وصفة محبوبة بالتدريب (بيرجعوا للـ Popularity): {no_profile.sum():,} من {n_eval:,} ({no_profile.mean()*100:.1f}%)")
print(f"وصفات محبوبة بالتدريب لكل مستخدمة: median {np.median(n_liked_tr):.0f}")


def content_scores_mean(rows):
    S = (P[rows] @ X_cat.T).toarray()
    fb = no_profile[rows]
    if fb.any():
        S[fb] = pop_score
    return S


recs_cb = recommend(content_scores_mean)
results.append(evaluate(recs_cb, "Content-Based A: mean profile"))
results[-1]

مستخدمات بدون ولا وصفة محبوبة بالتدريب (بيرجعوا للـ Popularity): 2 من 2,392 (0.1%)
وصفات محبوبة بالتدريب لكل مستخدمة: median 10


{'system': 'Content-Based A: mean profile',
 'HitRate@10': 0.0038,
 'NDCG@10': 0.0009,
 'Coverage_%': 31.68,
 'hit_users': 9,
 'cold_hits': 1,
 'eval_users': 'all'}

In [27]:
# B. أعلى تشابه: لكل مستخدمة، تشابه كل وصفة بالكتالوج مع كل وصفة حبّتها، ومناخد الأعلى
X_cat_T = X_cat.T.tocsc()


def content_scores_max(rows):
    S = np.empty((len(rows), n_cat), dtype=np.float32)
    for r_, row in enumerate(rows):
        liked_rows = A[row].indices                      # الوصفات اللي حبّتها (أسطر X)
        if len(liked_rows) == 0:
            S[r_] = pop_score
        else:
            S[r_] = (X[liked_rows] @ X_cat_T).max(axis=0).toarray().ravel()
    return S


recs_cb_max = recommend(content_scores_max)
results.append(evaluate(recs_cb_max, "Content-Based B: max-similarity"))
results[-1]

{'system': 'Content-Based B: max-similarity',
 'HitRate@10': 0.0063,
 'NDCG@10': 0.0016,
 'Coverage_%': 32.9,
 'hit_users': 15,
 'cold_hits': 0,
 'eval_users': 'all'}

### مثال لمستخدمة (فحص منطقي)

الوصفات اللي حبّتها بالتدريب، وتحتها اقتراحات الطريقتين. لازم يكون في تشابه واضح بالمكونات، وB لازم تكون أكتر تنوّع إذا ذوقها متنوّع.

In [28]:
ex_row = int(np.argmax(np.where(n_liked_tr <= 15, n_liked_tr, 0)))   # مستخدمة عندها عدد معقول من المحبوبة
ex_user = eval_users[ex_row]
liked_ids = lt.loc[lt["user_id"] == ex_user, "recipe_id"]
print(f"user {ex_user}: {len(liked_ids)} وصفة محبوبة بالتدريب")
print(recipes.set_index("RecipeId").loc[liked_ids, ["Name"]].head(10).to_string())
print("\nA. اقتراحات المتوسط:")
print(show_recs(recs_cb, ex_row).to_string())
print("\nB. اقتراحات أعلى تشابه:")
print(show_recs(recs_cb_max, ex_row).to_string())

user 12882: 15 وصفة محبوبة بالتدريب
                                                       Name
RecipeId                                                   
3094                           Confectioner's Sugar Cookies
8953                                Cream Cheese Pound Cake
9973                       Pineapple and Sour Cream Muffins
12736                                 Orange Glazed Chicken
14059                           Lennie's Special Shortbread
14467                                         Pecan Tassies
19542                                  Blue Cheese Dressing
21812                                   Peanut Butter Balls
24521                                         Blueberry Pie
26306     Copycat  Green Giant Niblets Corn in Butter Sauce

A. اقتراحات المتوسط:
   recipe_id                                    Name  hidden_hit
0     112802                        Pie Filling Bars       False
1     394718               Firecracker Fudge Cookies       False
2      86579        Melt in

## 9. النتائج

In [29]:
res = pd.DataFrame(results).set_index("system")
res

,HitRate@10,NDCG@10,Coverage_%,hit_users,cold_hits,eval_users
system,,,,,,
Random,0.0021,0.0005,85.60,5,0,all
Popularity,0.0134,0.0043,0.11,32,0,all
Content-Based A: mean profile,0.0038,0.0009,31.68,9,1,all
Content-Based B: max-similarity,0.0063,0.0016,32.90,15,0,all


In [30]:
# أكتر الوصفات تكراراً بالاقتراحات لكل نظام (لقلق popularity bias)
def most_repeated(recs, n=5):
    ids, cnt = np.unique(recs, return_counts=True)
    top = np.argsort(-cnt)[:n]
    return pd.DataFrame({"Name": recipes.set_index("RecipeId").loc[catalog_ids[ids[top]], "Name"].to_numpy(),
                         "users_%": (cnt[top] / n_eval * 100).round(1)})

for nm, rc in [("Popularity", recs_pop), ("Content-Based A (mean)", recs_cb), ("Content-Based B (max)", recs_cb_max)]:
    print(f"\n{nm}: أكتر 5 وصفات تكراراً (نسبة المستخدمات اللي طلعتلهن)")
    print(most_repeated(rc).to_string(index=False))


Popularity: أكتر 5 وصفات تكراراً (نسبة المستخدمات اللي طلعتلهن)
                                        Name  users_%
      Cream Cheese Icing (Mrs. Echard Icing)     99.8
              Italian Style Skillet Zucchini     99.7
        Shirley's Plain or Blueberry Muffins     99.7
                      Pumpkin Cinnamon Rolls     99.6
Sweet, Buttery Rolls  (Bread Machine Recipe)     99.6

Content-Based A (mean): أكتر 5 وصفات تكراراً (نسبة المستخدمات اللي طلعتلهن)
                      Name  users_%
      Cottage Pudding Cake     21.8
Sunday Morning Coffee Cake     17.8
 New York-Style Crumb Cake     17.7
   Buttercream Coffee Cake     17.0
        Snicker Doodle Pie     11.7

Content-Based B (max): أكتر 5 وصفات تكراراً (نسبة المستخدمات اللي طلعتلهن)
                                    Name  users_%
Phoebe's Fabulous Oatmeal-Raisin Cookies      7.6
                         Classic Muffins      6.2
                             Sugar bread      6.2
           Barbara Pratt's Tea  Biscuits  

**قراءة النتائج:**
- Popularity لازم يغلب Random بفرق كبير؛ إذا لأ في مشكلة بالإطار.
- Coverage تبع Popularity صغيرة جداً بالتصميم (نفس العشرة تقريباً للكل).
- Content-Based ممكن يطلع HitRate أقل من Popularity (هالشي شائع بداتا التقييمات)، بس Coverage أعلى بكتير ويقدر يلاقي cold items. هالفرق هو سبب الـ Hybrid.
- **A مقابل B:** منعتمد للتطبيق الطريقة اللي بتطلع أحسن بـ HitRate وNDCG، ومنشوف كمان الـ Coverage وجدول التكرار (هل اختفت الكيكات المتكررة؟).
- عمود `cold_hits` بيوري كم وصفة **ما إلها ولا تفاعل بالتدريب** لاقاها كل نظام. Popularity ما بيقدر يلاقي ولا وحدة.

## 10. الحفظ

In [31]:
pd.DataFrame({"recipe_id": catalog_ids}).to_csv(CATALOG_CSV, index=False, encoding="utf-8-sig")
test_cat.to_pickle(EVAL_TEST_PKL)
pd.DataFrame({"user_id": eval_users}).to_csv(EVAL_USERS_CSV, index=False, encoding="utf-8-sig")
sparse.save_npz(TFIDF_NPZ, X.tocsr())
pd.DataFrame({"recipe_id": recipes["RecipeId"]}).to_csv(TFIDF_IDS_CSV, index=False, encoding="utf-8-sig")
res.reset_index().to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
print("saved:", CATALOG_CSV, EVAL_TEST_PKL, EVAL_USERS_CSV, TFIDF_NPZ, TFIDF_IDS_CSV, RESULTS_CSV)

saved: catalog_ids.csv eval_test_catalog.pkl eval_users_sample.csv tfidf_ingredients.npz tfidf_recipe_ids.csv results_07.csv


## الخطوة الجاية (نوتبوك ٨)

الـ CF (User-Based وItem-Based، صريح وضمني) على **نفس** `eval_users_sample.csv` و`eval_test_catalog.pkl`، وبنفس دوال `recommend` و`evaluate`. قبلها منناقش: المقياس، عدد الجيران k، والـ cold items.

ملاحظة: `seen` و`relevant` و`A` محسوبين للمستخدمات بـ `eval_users`، فلازم النوتبوكات الجاية تقرأ نفس `eval_users_sample.csv`.